In [0]:
import pyspark.sql.functions as F

# ======================================================================================
# 0. PROJECT STANDARDS & CONTEXT FIX
# ======================================================================================
CATALOG = "vstone_catalog"
GOLD = "gold"
SECURITY = "security"

# FIX: Setting the current catalog context explicitly to avoid AnalysisException
print(f"📡 Setting catalog context to {CATALOG}...")
spark.sql(f"USE CATALOG {CATALOG}")

# ======================================================================================
# 1. SECURITY LAYER TESTS
# ======================================================================================

def run_security_unit_tests():
    print(f"🔍 Starting Security Layer Validation for {CATALOG}...\n")
    
    try:
        # --- TEST 1: RLS Function Existence ---
        # Ab Spark current catalog ke 'security' schema mein dhoondhega
        funcs = spark.sql(f"SHOW FUNCTIONS IN {SECURITY}").collect()
        func_names = [r[0].lower() for r in funcs]
        
        assert any("listing_region_filter" in f for f in func_names), \
            f"❌ FAIL: listing_region_filter RLS function not found in {SECURITY}!"
        print("✅ PASS: RLS function 'listing_region_filter' exists.")

        # --- TEST 2: Column Masking Functions ---
        assert any("mask_exact_price" in f for f in func_names), "❌ FAIL: mask_exact_price function missing!"
        assert any("mask_listing_id" in f for f in func_names), "❌ FAIL: mask_listing_id function missing!"
        print("✅ PASS: All Column Masking functions verified.")

        # --- TEST 3: Metadata Persistence ---
        # Full path use karna safe hai (catalog.schema.table)
        table_meta = spark.sql(f"DESCRIBE TABLE EXTENDED {GOLD}.fact_listings_liquid")
        comment_row = table_meta.filter(F.col("col_name") == "comment").collect()
        
        if comment_row and comment_row[0]['data_type']:
            print(f"✅ PASS: Metadata comment verified: '{comment_row[0]['data_type'][:40]}...'")
        else:
            print("⚠️ WARNING: Metadata comment missing on fact_listings_liquid!")

        # --- TEST 4: Principal Privilege Audit ---
        grants = spark.sql(f"SHOW GRANTS ON TABLE {GOLD}.fact_listings_liquid").collect()
        has_analyst_select = any(r['Principal'] == 'analyst_group' and r['ActionType'] == 'SELECT' for r in grants)
        
        assert has_analyst_select, "❌ FAIL: analyst_group does not have SELECT permission!"
        print("✅ PASS: SELECT privileges verified for 'analyst_group'.")

        print("\n🎯 ALL SECURITY TESTS PASSED: Gold Layer is production-ready.")

    except Exception as e:
        print(f"❌ CRITICAL ERROR during testing: {str(e)}")

# Execution
run_security_unit_tests()

In [0]:
# =========================================================
# FINAL AUDIT: Security Policies (Column & Row Level)
# =========================================================
import pyspark.sql.functions as F

CATALOG = "vstone_catalog"
GOLD_SCHEMA = "gold"
TABLE_NAME = "fact_listings_liquid"

print(f"📡 Auditing Security Layers for {CATALOG}.{GOLD_SCHEMA}.{TABLE_NAME}...")

# 1. DIRECT METADATA AUDIT (Shows Rows & Columns with Security)
# Describe Extended is the most robust way to see Masks and Filters
print("\n🔍 SECURITY POLICIES (Row Filters & Column Masks):")

describe_df = spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{GOLD_SCHEMA}.{TABLE_NAME}")

# Filtering specifically for security metadata rows
security_audit_df = describe_df.filter(
    (F.col("col_name").contains("Mask")) | 
    (F.col("col_name").contains("Row Filter"))
)

display(security_audit_df)

# 2. TARGET COLUMN VERIFICATION
# Checking status of specific columns we secured in Day 8-9
print(f"\n🔍 VERIFYING TARGET SECURE COLUMNS (listing_id, price_rub):")
column_audit_df = describe_df.filter(
    (F.col("col_name") == "listing_id") | 
    (F.col("col_name") == "price_rub") |
    (F.col("col_name") == "location_key")
)

display(column_audit_df)

# 3. GLOBAL ACCESS AUDIT (Separation of Concerns)
# Requirement: Analysts should only see Gold, access to Bronze/Silver must be revoked
print("\n🔍 CATALOG ACCESS CHECK (Ensuring Layer Isolation):")
try:
    print("Checking Bronze access (Expected to fail for analyst_group)...")
    spark.sql(f"SHOW TABLES IN {CATALOG}.bronze")
except Exception as e:
    print(f"✅ PASS: Access to Bronze is restricted: {str(e)[:50]}...")

print("\n🎯 Security Implementation Verified for Day 8-9!")

In [0]:
# =========================================================
# LIVE DEMO: SECURE COLUMNS (listing_id & price_rub)
# =========================================================
CATALOG = "vstone_catalog"
GOLD_SCHEMA = "gold"
TABLE = "fact_listings_liquid"

print(f"🕵️ Showing Secure View for: {CATALOG}.{GOLD_SCHEMA}.{TABLE}")

# Querying specifically for the columns where masking is applied
# Requirement: Non-admins see ID-***XXXX and Rounded Prices
secure_data_df = spark.sql(f"""
    SELECT 
        listing_id, 
        price_rub, 
        brand, 
        model, 
        location_key
    FROM {CATALOG}.{GOLD_SCHEMA}.{TABLE}
    LIMIT 10
""")

display(secure_data_df)

print("\n💡 Note: If you see full IDs and exact prices, it's because you are an ADMIN.")
print("💡 To see masking in action, impersonate a user from the 'analyst_group'.")